In [2]:
from dotenv import load_dotenv
import os
import boto3
import pandas as pd

load_dotenv()

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{os.environ['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com",
    aws_access_key_id=os.environ["R2_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["R2_SECRET_ACCESS_KEY"],
    region_name="auto",
)

bucket = "colombia-dengue-forecasting"

In [3]:
years = ["2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]

cases_data_frames = []

for year in years:
    key = f"raw/cases/Datos_{year}_210.xlsx"
    local_path = f"data/raw/cases/{year}.xlsx"

    s3.download_file(bucket, key, local_path)
    df = pd.read_excel(local_path)
    cases_data_frames.append(df)

In [4]:
cases_data_frames[0].info()    

<class 'pandas.DataFrame'>
RangeIndex: 100117 entries, 0 to 100116
Data columns (total 69 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   CONSECUTIVE                100117 non-null  int64  
 1   COD_EVE                    100117 non-null  int64  
 2   FEC_NOT                    100117 non-null  str    
 3   SEMANA                     100117 non-null  int64  
 4   ANO                        100117 non-null  int64  
 5   COD_PRE                    100117 non-null  int64  
 6   COD_SUB                    100117 non-null  int64  
 7   EDAD                       100117 non-null  int64  
 8   UNI_MED                    100117 non-null  int64  
 9   SEXO                       100117 non-null  str    
 10  COD_PAIS_O                 100117 non-null  int64  
 11  COD_DPTO_O                 100117 non-null  int64  
 12  COD_MUN_O                  100117 non-null  int64  
 13  AREA                       100117 non-nu

| Column | Meaning | Values / format |
|---|---|---|
| `COD_EVE` | SIVIGILA event code | `210` = dengue; `220` = severe dengue; `580` = dengue mortality, handled as a separate event |
| `ANO` | Epidemiological year | The years present in the dataset |
| `SEMANA` | Epidemiological week | Week number, generally `1`–`53`; use the week paired with `ANO` |
| `COD_DPTO_O` | Department where the case occurred | DANE DIVIPOLA department code; keep as text to preserve leading zeros |
| `COD_MUN_O` | Municipality where the case occurred | DANE DIVIPOLA municipality code; check whether this export stores the municipality component or the full code |
| `COD_PAIS_O` | Country where the case occurred | Country code; Colombia is commonly represented as `170` |
| `INI_SIN` | Date symptoms began | Date; parse using the format in your source file |
| `FEC_NOT` | Date the case was notified | Date; parse using the format in your source file |
| `TIP_CAS` | Initial case classification/type | Event- and year-specific code; check the dengue data dictionary and the values in your files |
| `AJUSTE` | Case adjustment or updated classification | Dengue protocol mentions `3` = confirmed by laboratory, `5` = confirmed by epidemiological link, `6` = discarded, and `D` = digitization error; verify other values by year |
| `CONSECUTIVE` | Sequential record identifier | Identifier, not a category; uniqueness is not guaranteed across separate yearly files |

In [5]:
cols_to_keep = ["CONSECUTIVE", "COD_EVE", "ANO", "SEMANA", "COD_DPTO_O", "COD_MUN_O", "COD_PAIS_O", "INI_SIN", "FEC_NOT", "TIP_CAS", "AJUSTE"]
for i in range(0,len(cases_data_frames)):
    cases_data_frames[i] = cases_data_frames[i][cols_to_keep]

cases_data_frames[0].head()

,CONSECUTIVE,COD_EVE,ANO,SEMANA,COD_DPTO_O,COD_MUN_O,COD_PAIS_O,INI_SIN,FEC_NOT,TIP_CAS,AJUSTE
0,713315,210,2016,21,5,1,170,25/05/2016 12:00:00 a. m.,1/06/2016 12:00:00 a. m.,3,0
1,713316,210,2016,3,5,1,170,18/01/2016 12:00:00 a. m.,21/01/2016 12:00:00 a. m.,3,0
2,713323,210,2016,2,5,1,170,15/01/2016 12:00:00 a. m.,23/01/2016 12:00:00 a. m.,2,0
3,713353,210,2016,22,5,1,170,1/06/2016 12:00:00 a. m.,11/06/2016 12:00:00 a. m.,3,0
4,713310,210,2016,25,5,1,170,20/06/2016 12:00:00 a. m.,25/06/2016 12:00:00 a. m.,2,0


| Column | Value | Meaning |
|---|---:|---|
| `TIP_CAS` | `2` | Probable case |
| `TIP_CAS` | `3` | Confirmed by laboratory |
| `TIP_CAS` | `5` | Confirmed by epidemiological link |
| `AJUSTE` | `0` | No adjustment / first report |
| `AJUSTE` | `3` | Adjustment confirming by laboratory |
| `AJUSTE` | `5` | Adjustment confirming by epidemiological link |
| `AJUSTE` | `7` | Other update to the record; not a case classification |

In [6]:
global_cases = pd.concat(cases_data_frames, axis=0, join='outer', ignore_index=True)

In [7]:
global_cases['COD_DPTO_O'].unique()

array([ 5, 85, 66, 50, 76, 18, 68, 20, 17, 41, 73, 15, 25, 63, 70, 19,  8,
       91, 81, 27, 54, 94, 23, 86, 13, 95, 99, 47, 52,  1, 88,  0, 44, 97])

In [8]:
# Only cases that ocurred in Colombia, identified with M49 code 170 1037761
global_cases = global_cases[global_cases['COD_PAIS_O'] == 170]
global_cases

,CONSECUTIVE,COD_EVE,ANO,SEMANA,COD_DPTO_O,COD_MUN_O,COD_PAIS_O,INI_SIN,FEC_NOT,TIP_CAS,AJUSTE
0,713315,210,2016,21,5,1,170,25/05/2016 12:00:00 a. m.,1/06/2016 12:00:00 a. m.,3,0
1,713316,210,2016,3,5,1,170,18/01/2016 12:00:00 a. m.,21/01/2016 12:00:00 a. m.,3,0
2,713323,210,2016,2,5,1,170,15/01/2016 12:00:00 a. m.,23/01/2016 12:00:00 a. m.,2,0
3,713353,210,2016,22,5,1,170,1/06/2016 12:00:00 a. m.,11/06/2016 12:00:00 a. m.,3,0
4,713310,210,2016,25,5,1,170,20/06/2016 12:00:00 a. m.,25/06/2016 12:00:00 a. m.,2,0
...,...,...,...,...,...,...,...,...,...,...,...
1040490,4754,210,2025,51,68,689,170,18/12/2025 12:00:00 a. m.,19/12/2025 12:00:00 a. m.,3,0
1040491,4716,210,2025,3,8,758,170,12/01/2025 12:00:00 a. m.,17/01/2025 12:00:00 a. m.,3,3
1040492,4655,210,2025,50,47,1,170,8/12/2025 12:00:00 a. m.,12/12/2025 12:00:00 a. m.,3,0
1040493,1974,210,2025,28,73,1,170,10/07/2025 12:00:00 a. m.,14/07/2025 12:00:00 a. m.,2,7


In [9]:
for col in cols_to_keep:
    print(col)
    print(global_cases[col].isnull().sum())

CONSECUTIVE
0
COD_EVE
0
ANO
0
SEMANA
0
COD_DPTO_O
0
COD_MUN_O
0
COD_PAIS_O
0
INI_SIN
24
FEC_NOT
0
TIP_CAS
0
AJUSTE
0


| Código | Departamento | Código | Departamento |
|---:|---|---:|---|
| 05 | Antioquia | 52 | Nariño |
| 08 | Atlántico | 54 | Norte de Santander |
| 11 | Bogotá, D.C. | 63 | Quindío |
| 13 | Bolívar | 66 | Risaralda |
| 15 | Boyacá | 68 | Santander |
| 17 | Caldas | 70 | Sucre |
| 18 | Caquetá | 73 | Tolima |
| 19 | Cauca | 76 | Valle del Cauca |
| 20 | Cesar | 81 | Arauca |
| 23 | Córdoba | 85 | Casanare |
| 25 | Cundinamarca | 86 | Putumayo |
| 27 | Chocó | 88 | San Andrés y Providencia |
| 41 | Huila | 91 | Amazonas |
| 44 | La Guajira | 94 | Guainía |
| 47 | Magdalena | 95 | Guaviare |
| 50 | Meta | 97 | Vaupés |
|  |  | 99 | Vichada |

In [10]:
department_codes = global_cases["COD_DPTO_O"].value_counts().index.tolist()
department_codes_df = pd.DataFrame(data=department_codes, columns=['COD_DPTO_O'])
department_codes_df = department_codes_df[department_codes_df['COD_DPTO_O'] != 0]
department_codes_df

,COD_DPTO_O
0,76
1,68
2,73
3,5
4,50
5,13
6,41
7,8
8,54
9,23


In [11]:
global_cases = global_cases[global_cases["COD_DPTO_O"] != 0]
global_cases.shape[0]

1037727

In [12]:
keys = ['ANO', 'SEMANA', 'COD_DPTO_O']
counts_by_week_year_department = global_cases.groupby(keys).size().rename('case_count').reset_index()
counts_by_week_year_department

,ANO,SEMANA,COD_DPTO_O,case_count
0,2016,1,5,391
1,2016,1,8,128
2,2016,1,13,69
3,2016,1,15,21
4,2016,1,17,25
...,...,...,...,...
15400,2025,53,85,6
15401,2025,53,86,9
15402,2025,53,91,12
15403,2025,53,95,4


In [13]:

import numpy as np

epidemological_weeks_per_year = [(2016, 52),(2017, 52),(2018, 52),(2019, 52),(2020, 53),(2021, 52),(2022, 52),(2023, 52),(2024, 52),(2025, 53)]

year_week = np.array([(year , week) for year, num_weeks in epidemological_weeks_per_year for week in range(1, num_weeks + 1)])
year_week_df = pd.DataFrame(data = year_week, columns=['ANO', 'SEMANA'])
year_week_df

,ANO,SEMANA
0,2016,1
1,2016,2
2,2016,3
3,2016,4
4,2016,5
...,...,...
517,2025,49
518,2025,50
519,2025,51
520,2025,52


In [14]:
department_year_week_df = year_week_df.merge(department_codes_df, how='cross')
department_year_week_df

,ANO,SEMANA,COD_DPTO_O
0,2016,1,76
1,2016,1,68
2,2016,1,73
3,2016,1,5
4,2016,1,50
...,...,...,...
16699,2025,53,95
16700,2025,53,99
16701,2025,53,88
16702,2025,53,94


In [15]:
weekly_cases_complete_df = department_year_week_df.merge(counts_by_week_year_department, on=keys, how='left').fillna({'case_count' : 0})
weekly_cases_complete_df['case_count'] = weekly_cases_complete_df['case_count'].astype(int)
weekly_cases_complete_df

,ANO,SEMANA,COD_DPTO_O,case_count
0,2016,1,76,558
1,2016,1,68,148
2,2016,1,73,269
3,2016,1,5,391
4,2016,1,50,57
...,...,...,...,...
16699,2025,53,95,4
16700,2025,53,99,3
16701,2025,53,88,0
16702,2025,53,94,0


In [16]:
print(weekly_cases_complete_df['COD_DPTO_O'].unique())

[76 68 73  5 50 13 41  8 54 23 25 20 70 85 47 19 86 44 66 63 18 81 52 27
 15 17 91 95 99 88 94 97]


In [7]:
from zipfile import ZipFile
import xarray as xr

with ZipFile('./data/raw/2m_temperature/temp.zip') as z:
    z.extract('data.grib', './data/raw/2m_temperature')

dataset = xr.open_dataset(
    './data/raw/2m_temperature/data.grib',
    engine='cfgrib',
    chunks={'time': 24}
)

print(dataset)

Ignoring index file './data/raw/2m_temperature/data.grib.5b7b6.idx' older than GRIB file


<xarray.Dataset> Size: 2GB
Dimensions:     (time: 87672, latitude: 77, longitude: 69)
Coordinates:
  * time        (time) datetime64[ns] 701kB 2016-01-01 ... 2025-12-31T23:00:00
    valid_time  (time) datetime64[ns] 701kB dask.array<chunksize=(24,), meta=np.ndarray>
  * latitude    (latitude) float64 616B 14.0 13.75 13.5 ... -4.5 -4.75 -5.0
  * longitude   (longitude) float64 552B -83.0 -82.75 -82.5 ... -66.25 -66.0
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Data variables:
    t2m         (time, latitude, longitude) float32 2GB dask.array<chunksize=(24, 77, 69), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-12T21:44 GRIB to

In [5]:
import geopandas as gpd

department_polygons = gpd.read_file('./data/map_polygons/department_polygons.geojson').to_crs('EPSG:4326')
department_polygons

,dpto_ccdgo,dpto_cnmbr,dpto_ano_c,dpto_act_a,dpto_narea,dpto_nano,shape_Leng,shape_Area,geometry
0,05,ANTIOQUIA,1886,Constitucion Politica de 1886,62806.982729,2024,21.479346,5.135102,"MULTIPOLYGON (((-76.41355 8.87383, -76.40465 8..."
1,08,ATLÁNTICO,1910,Ley 21 de 1910,3314.560851,2024,2.569609,0.273832,"MULTIPOLYGON (((-74.84946 11.09778, -74.84938 ..."
2,11,"BOGOTÁ, D.C.",1538,Constitucion Politica de 1886,1622.852979,2024,3.765371,0.132208,"MULTIPOLYGON (((-74.07059 4.82856, -74.07036 4..."
3,13,BOLÍVAR,1886,Constitucion Politica de 1886,26720.306964,2024,16.272240,2.195667,"MULTIPOLYGON (((-76.17318 9.38785, -76.17287 9..."
4,15,BOYACÁ,1886,Constitucion Politica de 1886,23137.997637,2024,15.906767,1.888387,"MULTIPOLYGON (((-72.17368 7.05308, -72.17277 7..."
5,17,CALDAS,1905,11 de Abril de 1905,7425.220161,2024,6.664502,0.605498,"MULTIPOLYGON (((-74.67154 5.77147, -74.67054 5..."
6,18,CAQUETÁ,1981,Ley 78 del 29 de Diciembre de 1981,92831.282595,2024,21.218640,7.540241,"MULTIPOLYGON (((-76.09066 1.07012, -76.09066 1..."
7,19,CAUCA,1857,15 de junio de 1857,31242.864721,2024,13.953178,2.534415,"MULTIPOLYGON (((-76.45922 3.32872, -76.45878 3..."
8,20,CESAR,1967,Ley 25 21 de junio de 1967,22534.251126,2024,12.748366,1.855656,"MULTIPOLYGON (((-73.45335 10.86765, -73.45318 ..."
9,23,CÓRDOBA,1951,Ley 9 del 18 de Diciembre de 1951,25086.239945,2024,9.711373,2.057508,"MULTIPOLYGON (((-75.88119 9.42363, -75.8794 9...."


In [8]:
dataset.latitude.values

array([14.  , 13.75, 13.5 , 13.25, 13.  , 12.75, 12.5 , 12.25, 12.  ,
       11.75, 11.5 , 11.25, 11.  , 10.75, 10.5 , 10.25, 10.  ,  9.75,
        9.5 ,  9.25,  9.  ,  8.75,  8.5 ,  8.25,  8.  ,  7.75,  7.5 ,
        7.25,  7.  ,  6.75,  6.5 ,  6.25,  6.  ,  5.75,  5.5 ,  5.25,
        5.  ,  4.75,  4.5 ,  4.25,  4.  ,  3.75,  3.5 ,  3.25,  3.  ,
        2.75,  2.5 ,  2.25,  2.  ,  1.75,  1.5 ,  1.25,  1.  ,  0.75,
        0.5 ,  0.25,  0.  , -0.25, -0.5 , -0.75, -1.  , -1.25, -1.5 ,
       -1.75, -2.  , -2.25, -2.5 , -2.75, -3.  , -3.25, -3.5 , -3.75,
       -4.  , -4.25, -4.5 , -4.75, -5.  ])

In [9]:
import numpy as np

lat_grid , lon_grid = np.meshgrid(dataset.latitude.values, dataset.longitude.values, indexing='ij')
lat_indeces, lon_indeces = np.indices(lat_grid.shape)
lat_indeces

array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 1,  1,  1, ...,  1,  1,  1],
       [ 2,  2,  2, ...,  2,  2,  2],
       ...,
       [74, 74, 74, ..., 74, 74, 74],
       [75, 75, 75, ..., 75, 75, 75],
       [76, 76, 76, ..., 76, 76, 76]], shape=(77, 69))

In [10]:
lat_grid

array([[14.  , 14.  , 14.  , ..., 14.  , 14.  , 14.  ],
       [13.75, 13.75, 13.75, ..., 13.75, 13.75, 13.75],
       [13.5 , 13.5 , 13.5 , ..., 13.5 , 13.5 , 13.5 ],
       ...,
       [-4.5 , -4.5 , -4.5 , ..., -4.5 , -4.5 , -4.5 ],
       [-4.75, -4.75, -4.75, ..., -4.75, -4.75, -4.75],
       [-5.  , -5.  , -5.  , ..., -5.  , -5.  , -5.  ]], shape=(77, 69))

In [11]:
grid_pts = gpd.GeoDataFrame({
    "lat_ind" : lat_indeces.ravel(),
    "lon_ind" : lon_indeces.ravel()
},
geometry=gpd.points_from_xy(lon_grid.ravel(), lat_grid.ravel()),
crs="EPSG:4326")
grid_pts

,lat_ind,lon_ind,geometry
0,0,0,POINT (-83 14)
1,0,1,POINT (-82.75 14)
2,0,2,POINT (-82.5 14)
3,0,3,POINT (-82.25 14)
4,0,4,POINT (-82 14)
...,...,...,...
5308,76,64,POINT (-67 -5)
5309,76,65,POINT (-66.75 -5)
5310,76,66,POINT (-66.5 -5)
5311,76,67,POINT (-66.25 -5)


In [12]:
points_with_departments = gpd.sjoin(grid_pts, department_polygons[['dpto_ccdgo', 'dpto_cnmbr', 'geometry']], how='left', predicate='within')
points_with_departments

,lat_ind,lon_ind,geometry,index_right,dpto_ccdgo,dpto_cnmbr
0,0,0,POINT (-83 14),NaN,NaN,NaN
1,0,1,POINT (-82.75 14),NaN,NaN,NaN
2,0,2,POINT (-82.5 14),NaN,NaN,NaN
3,0,3,POINT (-82.25 14),NaN,NaN,NaN
4,0,4,POINT (-82 14),NaN,NaN,NaN
...,...,...,...,...,...,...
5308,76,64,POINT (-67 -5),NaN,NaN,NaN
5309,76,65,POINT (-66.75 -5),NaN,NaN,NaN
5310,76,66,POINT (-66.5 -5),NaN,NaN,NaN
5311,76,67,POINT (-66.25 -5),NaN,NaN,NaN


In [13]:
points_with_departments = points_with_departments.dropna(subset='dpto_ccdgo')
points_with_departments

,lat_ind,lon_ind,geometry,index_right,dpto_ccdgo,dpto_cnmbr
528,7,45,POINT (-71.75 12.25),13.0,44,LA GUAJIRA
529,7,46,POINT (-71.5 12.25),13.0,44,LA GUAJIRA
530,7,47,POINT (-71.25 12.25),13.0,44,LA GUAJIRA
596,8,44,POINT (-72 12),13.0,44,LA GUAJIRA
597,8,45,POINT (-71.75 12),13.0,44,LA GUAJIRA
...,...,...,...,...,...,...
4882,70,52,POINT (-70 -3.5),28.0,91,AMAZONAS
4949,71,50,POINT (-70.5 -3.75),28.0,91,AMAZONAS
4950,71,51,POINT (-70.25 -3.75),28.0,91,AMAZONAS
4951,71,52,POINT (-70 -3.75),28.0,91,AMAZONAS


In [25]:
points_with_departments.groupby(['dpto_ccdgo', 'dpto_cnmbr']).size().sort_values()

dpto_ccdgo  dpto_cnmbr        
11          BOGOTÁ, D.C.            1
63          QUINDIO                 2
08          ATLÁNTICO               3
66          RISARALDA               6
17          CALDAS                  8
70          SUCRE                  13
41          HUILA                  24
76          VALLE DEL CAUCA        26
25          CUNDINAMARCA           27
44          LA GUAJIRA             29
54          NORTE DE SANTANDER     30
20          CESAR                  31
47          MAGDALENA              32
23          CÓRDOBA                32
81          ARAUCA                 33
73          TOLIMA                 33
86          PUTUMAYO               34
15          BOYACÁ                 34
13          BOLÍVAR                35
52          NARIÑO                 40
68          SANTANDER              40
19          CAUCA                  42
85          CASANARE               57
27          CHOCÓ                  61
97          VAUPÉS                 70
95          GUAVIAR

In [26]:
lat_i = xr.DataArray(points_with_departments['lat_ind'].to_numpy(dtype='int64'), dims='points')
lon_i = xr.DataArray(points_with_departments['lon_ind'].to_numpy(dtype='int64'), dims='points')

points_temp = dataset['t2m'].isel(latitude=lat_i, longitude=lon_i)

In [27]:
points_temp = points_temp.assign_coords(department=('points', points_with_departments['dpto_ccdgo'].astype(str).to_numpy()))
points_temp

<xarray.DataArray 't2m' (time: 87672, points: 1493)> Size: 524MB
dask.array<transpose, shape=(87672, 1493), dtype=float32, chunksize=(24, 1493), chunktype=numpy.ndarray>
Coordinates:
  * time        (time) datetime64[ns] 701kB 2016-01-01 ... 2025-12-31T23:00:00
    valid_time  (time) datetime64[ns] 701kB dask.array<chunksize=(24,), meta=np.ndarray>
    latitude    (points) float64 12kB 12.25 12.25 12.25 ... -3.75 -3.75 -4.0
    longitude   (points) float64 12kB -71.75 -71.5 -71.25 ... -70.25 -70.0 -70.0
    department  (points) object 12kB '44' '44' '44' '44' ... '91' '91' '91' '91'
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Dimensions without coordinates: points
Attributes: (12/31)
    GRIB_paramId:                             167
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           2t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre temperature
    units:                                    K
    standard_name:                            unknown

In [28]:
hourly_by_department = (points_temp - 273.15).groupby('department').mean('points').rename('2m_temperature')
hourly_by_department

<xarray.DataArray '2m_temperature' (time: 87672, department: 32)> Size: 11MB
dask.array<transpose, shape=(87672, 32), dtype=float32, chunksize=(24, 1), chunktype=numpy.ndarray>
Coordinates:
  * time        (time) datetime64[ns] 701kB 2016-01-01 ... 2025-12-31T23:00:00
    valid_time  (time) datetime64[ns] 701kB dask.array<chunksize=(24,), meta=np.ndarray>
  * department  (department) object 256B '05' '08' '11' '13' ... '95' '97' '99'
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Attributes: (12/31)
    GRIB_paramId:                             167
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           2t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre temperature
    units:                                    K
    standard_name:                            unknown

In [29]:
daily_by_department = hourly_by_department.resample(time='D').mean()
daily_by_department

<xarray.DataArray '2m_temperature' (time: 3653, department: 32)> Size: 468kB
dask.array<stack, shape=(3653, 32), dtype=float32, chunksize=(1, 1), chunktype=numpy.ndarray>
Coordinates:
  * time        (time) datetime64[ns] 29kB 2016-01-01 2016-01-02 ... 2025-12-31
  * department  (department) object 256B '05' '08' '11' '13' ... '95' '97' '99'
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Attributes: (12/31)
    GRIB_paramId:                             167
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           2t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre temperature
    units:                                    K
    standard_name:                            unknown

In [30]:
import pandas as pd

def epi_calendar(start, end):
    days = pd.date_range(start, end, freq='D')
    sun = days - pd.to_timedelta((days.day_of_week + 1) % 7, unit='D')
    year = (sun + pd.Timedelta(days=3)).year

    def first_sunday(y):
        jan_1 = pd.Timestamp(y, 1, 1)
        s = jan_1 - pd.to_timedelta((jan_1.day_of_week + 1) % 7, unit='D')
        return s if (s + pd.Timedelta(days=3)).year == y else s + pd.Timedelta(days=7)
    
    first = pd.Series({ y: first_sunday(y) for y in set(year) })
    week = ((sun - first.loc[year].values).days // 7) + 1
    return pd.DataFrame({'day':days, 'epi_week': week, 'epi_year': year})

epi_year_week_per_day_df = epi_calendar('2016-01-01', '2025-12-31')
epi_year_week_per_day_df


,day,epi_week,epi_year
0,2016-01-01,52,2015
1,2016-01-02,52,2015
2,2016-01-03,1,2016
3,2016-01-04,1,2016
4,2016-01-05,1,2016
...,...,...,...
3648,2025-12-27,52,2025
3649,2025-12-28,53,2025
3650,2025-12-29,53,2025
3651,2025-12-30,53,2025


In [31]:
epi_year_week_per_day_df = epi_year_week_per_day_df.set_index('day')

times = daily_by_department.get_index('time')

daily_by_department = daily_by_department.assign_coords(
    epi_week = ('time', epi_year_week_per_day_df.loc[times, 'epi_week'].values),
    epi_year = ('time', epi_year_week_per_day_df.loc[times, 'epi_year'].values)
)

daily_by_department


<xarray.DataArray '2m_temperature' (time: 3653, department: 32)> Size: 468kB
dask.array<stack, shape=(3653, 32), dtype=float32, chunksize=(1, 1), chunktype=numpy.ndarray>
Coordinates:
  * time        (time) datetime64[ns] 29kB 2016-01-01 2016-01-02 ... 2025-12-31
    epi_week    (time) int64 29kB 52 52 1 1 1 1 1 1 ... 52 52 52 52 53 53 53 53
    epi_year    (time) int32 15kB 2015 2015 2016 2016 ... 2025 2025 2025 2025
  * department  (department) object 256B '05' '08' '11' '13' ... '95' '97' '99'
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Attributes: (12/31)
    GRIB_paramId:                             167
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           2t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre temperature
    units:                                    K
    standard_name:                            unknown

In [32]:
weekly_temp_by_department = daily_by_department.groupby(['epi_week', 'epi_year']).mean('time')
weekly_temp_by_department

/opt/homebrew/Caskroom/miniforge/base/envs/dengue_venv/lib/python3.12/site-packages/dask/array/core.py:5201: PerformanceWarning: Increasing number of chunks by factor of 32
  result = blockwise(


<xarray.DataArray '2m_temperature' (department: 32, epi_week: 53, epi_year: 11)> Size: 75kB
dask.array<reshape, shape=(32, 53, 11), dtype=float32, chunksize=(1, 1, 1), chunktype=numpy.ndarray>
Coordinates:
  * department  (department) object 256B '05' '08' '11' '13' ... '95' '97' '99'
  * epi_week    (epi_week) int64 424B 1 2 3 4 5 6 7 8 ... 47 48 49 50 51 52 53
  * epi_year    (epi_year) int32 44B 2015 2016 2017 2018 ... 2022 2023 2024 2025
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Attributes: (12/31)
    GRIB_paramId:                             167
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           2t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre temperature
    units:                                    K
    standard_name:                            unknown

In [33]:
weekly_cases_complete_df

,ANO,SEMANA,COD_DPTO_O,case_count
0,2016,1,76,558
1,2016,1,68,148
2,2016,1,73,269
3,2016,1,5,391
4,2016,1,50,57
...,...,...,...,...
16699,2025,53,95,4
16700,2025,53,99,3
16701,2025,53,88,0
16702,2025,53,94,0


In [34]:
weekly_cases_complete_df['COD_DPTO_O'].unique()

array([76, 68, 73,  5, 50, 13, 41,  8, 54, 23, 25, 20, 70, 85, 47, 19, 86,
       44, 66, 63, 18, 81, 52, 27, 15, 17, 91, 95, 99, 88, 94, 97])

In [35]:
weekly_cases_complete_df = weekly_cases_complete_df[~weekly_cases_complete_df['COD_DPTO_O'].isin([88])]

In [36]:
weekly_cases_complete_df['COD_DPTO_O'] = weekly_cases_complete_df['COD_DPTO_O'].astype(str).str.zfill(2)

In [37]:
weekly_temp_by_department_df = weekly_temp_by_department.to_dataframe().reset_index()
weekly_temp_by_department_df

,department,epi_week,epi_year,number,step,surface,2m_temperature
0,05,1,2015,0,0 days,0.0,NaN
1,05,1,2016,0,0 days,0.0,23.980185
2,05,1,2017,0,0 days,0.0,23.351353
3,05,1,2018,0,0 days,0.0,22.451874
4,05,1,2019,0,0 days,0.0,24.306385
...,...,...,...,...,...,...,...
18651,99,53,2021,0,0 days,0.0,NaN
18652,99,53,2022,0,0 days,0.0,NaN
18653,99,53,2023,0,0 days,0.0,NaN
18654,99,53,2024,0,0 days,0.0,NaN


In [38]:
weekly_temp_by_department_df = weekly_temp_by_department_df[weekly_temp_by_department_df['2m_temperature'].notnull() & 
                                                            ~weekly_temp_by_department_df['department'].str.contains('11') &
                                                            ~weekly_temp_by_department_df['epi_year'].isin([2015])]
weekly_temp_by_department_df = weekly_temp_by_department_df.drop(['step','number','surface'], axis=1)
weekly_temp_by_department_df

,department,epi_week,epi_year,2m_temperature
1,05,1,2016,23.980185
2,05,1,2017,23.351353
3,05,1,2018,22.451874
4,05,1,2019,24.306385
5,05,1,2020,23.357378
...,...,...,...,...
18642,99,52,2023,26.947048
18643,99,52,2024,26.939646
18644,99,52,2025,27.782576
18650,99,53,2020,26.083239


In [39]:
weekly_temp_by_department_df.info()

<class 'pandas.DataFrame'>
Index: 16182 entries, 1 to 18655
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   department      16182 non-null  str    
 1   epi_week        16182 non-null  int64  
 2   epi_year        16182 non-null  int32  
 3   2m_temperature  16182 non-null  float32
dtypes: float32(1), int32(1), int64(1), str(1)
memory usage: 539.3 KB


In [40]:
weekly_cases_complete_df.info()

<class 'pandas.DataFrame'>
Index: 16182 entries, 0 to 16703
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   ANO         16182 non-null  int64
 1   SEMANA      16182 non-null  int64
 2   COD_DPTO_O  16182 non-null  str  
 3   case_count  16182 non-null  int64
dtypes: int64(3), str(1)
memory usage: 663.7 KB


In [41]:
weekly_temp_cases_by_department_df = weekly_cases_complete_df.merge(weekly_temp_by_department_df, 
                                                                    how='left', 
                                                                    left_on=['ANO','SEMANA','COD_DPTO_O'],
                                                                    right_on=['epi_year', 'epi_week', 'department'])
weekly_temp_cases_by_department_df

,ANO,SEMANA,COD_DPTO_O,case_count,department,epi_week,epi_year,2m_temperature
0,2016,1,76,558,76,1,2016,22.074368
1,2016,1,68,148,68,1,2016,22.576899
2,2016,1,73,269,73,1,2016,21.558670
3,2016,1,05,391,05,1,2016,23.980185
4,2016,1,50,57,50,1,2016,26.736807
...,...,...,...,...,...,...,...,...
16177,2025,53,91,12,91,53,2025,24.900299
16178,2025,53,95,4,95,53,2025,25.376633
16179,2025,53,99,3,99,53,2025,27.129372
16180,2025,53,94,0,94,53,2025,25.722433


In [42]:
with ZipFile('./data/raw/precipitation/precip.zip') as x:
    x.extract('data.grib', './data/raw/precipitation')

precipitation_dataset = xr.open_dataset(
    './data/raw/precipitation/data.grib',
    engine='cfgrib',
    chunks={'time': 24}
)

print(precipitation_dataset)

Ignoring index file './data/raw/precipitation/data.grib.5b7b6.idx' older than GRIB file


<xarray.Dataset> Size: 2GB
Dimensions:     (time: 7307, step: 12, latitude: 77, longitude: 69)
Coordinates:
  * time        (time) datetime64[ns] 58kB 2015-12-31T18:00:00 ... 2025-12-31...
  * step        (step) timedelta64[ns] 96B 01:00:00 02:00:00 ... 12:00:00
    valid_time  (time, step) datetime64[ns] 701kB dask.array<chunksize=(24, 12), meta=np.ndarray>
  * latitude    (latitude) float64 616B 14.0 13.75 13.5 ... -4.5 -4.75 -5.0
  * longitude   (longitude) float64 552B -83.0 -82.75 -82.5 ... -66.25 -66.0
    number      int64 8B ...
    surface     float64 8B ...
Data variables:
    tp          (time, step, latitude, longitude) float32 2GB dask.array<chunksize=(24, 12, 77, 69), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weathe

In [43]:
tp = precipitation_dataset["tp"]

n_missing = int(tp.isnull().sum().compute())
n_total = tp.size

print(f"{n_missing:,} of {n_total:,} values missing "
      f"({n_missing / n_total:.2%})")

63,756 of 465,865,092 values missing (0.01%)


In [44]:
points_precipitation = tp.isel(latitude=lat_i, longitude=lon_i)

points_precipitation

<xarray.DataArray 'tp' (time: 7307, step: 12, points: 1493)> Size: 524MB
dask.array<transpose, shape=(7307, 12, 1493), dtype=float32, chunksize=(24, 12, 1493), chunktype=numpy.ndarray>
Coordinates:
  * time        (time) datetime64[ns] 58kB 2015-12-31T18:00:00 ... 2025-12-31...
  * step        (step) timedelta64[ns] 96B 01:00:00 02:00:00 ... 12:00:00
    valid_time  (time, step) datetime64[ns] 701kB dask.array<chunksize=(24, 12), meta=np.ndarray>
    latitude    (points) float64 12kB 12.25 12.25 12.25 ... -3.75 -3.75 -4.0
    longitude   (points) float64 12kB -71.75 -71.5 -71.25 ... -70.25 -70.0 -70.0
    number      int64 8B ...
    surface     float64 8B ...
Dimensions without coordinates: points
Attributes: (12/31)
    GRIB_paramId:                             228
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            accum
    ...                                       ...
    GRIB_shortName:                           tp
    GRIB_totalNumber:                         0
    GRIB_units:                               m
    long_name:                                Total precipitation
    units:                                    m
    standard_name:                            unknown

In [45]:
hourly_precipitation_by_points = (points_precipitation.stack(t=("time", "step"))
              .reset_index("t", drop=True)
              .swap_dims(t="valid_time")
              .sortby("valid_time"))

daily_precipitation_by_points = hourly_precipitation_by_points.resample(valid_time='1D').sum()
daily_precipitation_by_points

<xarray.DataArray 'tp' (points: 1493, valid_time: 3655)> Size: 22MB
dask.array<transpose, shape=(1493, 3655), dtype=float32, chunksize=(1493, 1), chunktype=numpy.ndarray>
Coordinates:
    latitude    (points) float64 12kB 12.25 12.25 12.25 ... -3.75 -3.75 -4.0
    longitude   (points) float64 12kB -71.75 -71.5 -71.25 ... -70.25 -70.0 -70.0
  * valid_time  (valid_time) datetime64[ns] 29kB 2015-12-31 ... 2026-01-01
    number      int64 8B ...
    surface     float64 8B ...
Dimensions without coordinates: points
Attributes: (12/31)
    GRIB_paramId:                             228
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            accum
    ...                                       ...
    GRIB_shortName:                           tp
    GRIB_totalNumber:                         0
    GRIB_units:                               m
    long_name:                                Total precipitation
    units:                                    m
    standard_name:                            unknown

In [46]:
int(daily_precipitation_by_points.isnull().sum().compute())

0

In [47]:
coords_to_join_on = ['latitude', 'longitude']

left = daily_precipitation_by_points.set_index(points=coords_to_join_on)
right = points_temp.set_index(points=coords_to_join_on)

c = right['department'].reset_coords(drop=True)

left, c = xr.align(left, c, join='inner')

daily_precipitation_by_department = left.assign_coords(department=c)
daily_precipitation_by_department = daily_precipitation_by_department.reset_index('points')
daily_precipitation_by_department


<xarray.DataArray 'tp' (points: 1493, valid_time: 3655)> Size: 22MB
dask.array<transpose, shape=(1493, 3655), dtype=float32, chunksize=(1493, 1), chunktype=numpy.ndarray>
Coordinates:
    latitude    (points) float64 12kB 12.25 12.25 12.25 ... -3.75 -3.75 -4.0
    longitude   (points) float64 12kB -71.75 -71.5 -71.25 ... -70.25 -70.0 -70.0
    department  (points) object 12kB '44' '44' '44' '44' ... '91' '91' '91' '91'
  * valid_time  (valid_time) datetime64[ns] 29kB 2015-12-31 ... 2026-01-01
    number      int64 8B ...
    surface     float64 8B ...
Dimensions without coordinates: points
Attributes: (12/31)
    GRIB_paramId:                             228
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            accum
    ...                                       ...
    GRIB_shortName:                           tp
    GRIB_totalNumber:                         0
    GRIB_units:                               m
    long_name:                                Total precipitation
    units:                                    m
    standard_name:                            unknown

In [48]:
print(int(daily_precipitation_by_department.isnull().sum().compute()))

0


In [49]:
cell_weights = np.cos(np.deg2rad(daily_precipitation_by_department['latitude']))
department_codes = np.unique(daily_precipitation_by_department['department'].values)

weighted_department_means = []

for code in department_codes:
    is_in_department = daily_precipitation_by_department['department'] == code

    department_cells = daily_precipitation_by_department.where(is_in_department, drop=True)
    department_weights = cell_weights.where(is_in_department, drop=True)

    weighted_department_mean = department_cells.weighted(department_weights).mean(dim='points')

    weighted_department_mean = weighted_department_mean.expand_dims(department=[code])

    weighted_department_means.append(weighted_department_mean)

daily_avg_p_by_department = xr.concat(weighted_department_means, dim='department')
daily_avg_p_by_department = daily_avg_p_by_department.sel(valid_time=slice('2016-01-01', '2025-12-31'))
daily_avg_p_by_department

<xarray.DataArray 'tp' (department: 32, valid_time: 3653)> Size: 935kB
dask.array<getitem, shape=(32, 3653), dtype=float64, chunksize=(1, 1), chunktype=numpy.ndarray>
Coordinates:
  * department  (department) object 256B '05' '08' '11' '13' ... '95' '97' '99'
  * valid_time  (valid_time) datetime64[ns] 29kB 2016-01-01 ... 2025-12-31
    number      int64 8B 0
    surface     float64 8B 0.0
Attributes: (12/31)
    GRIB_paramId:                             228
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            accum
    ...                                       ...
    GRIB_shortName:                           tp
    GRIB_totalNumber:                         0
    GRIB_units:                               m
    long_name:                                Total precipitation
    units:                                    m
    standard_name:                            unknown

In [50]:
times = daily_avg_p_by_department.get_index('valid_time')
calendar = epi_year_week_per_day_df.reindex(times)

if calendar[['epi_year','epi_week']].isna().any().any():
    raise ValueError("some precipitation dates are missing ther epi labels")

daily_avg_p_by_department = daily_avg_p_by_department.assign_coords(
    epi_week = ('valid_time', epi_year_week_per_day_df.loc[times, 'epi_week'].values),
    epi_year = ('valid_time', epi_year_week_per_day_df.loc[times, 'epi_year'].values)
)

In [51]:
daily_avg_p_by_department = daily_avg_p_by_department.where((daily_avg_p_by_department['epi_year'] >= 2016) &
                                                            (daily_avg_p_by_department['epi_year'] <= 2025), 
                                                            drop=True)

In [52]:
weekly_avg_p_by_department = daily_avg_p_by_department.groupby(['epi_year','epi_week']).sum(dim='valid_time')
weekly_avg_p_by_department

/opt/homebrew/Caskroom/miniforge/base/envs/dengue_venv/lib/python3.12/site-packages/dask/array/core.py:5201: PerformanceWarning: Increasing number of chunks by factor of 32
  result = blockwise(


<xarray.DataArray 'tp' (department: 32, epi_year: 10, epi_week: 53)> Size: 136kB
dask.array<reshape, shape=(32, 10, 53), dtype=float64, chunksize=(1, 1, 1), chunktype=numpy.ndarray>
Coordinates:
  * department  (department) object 256B '05' '08' '11' '13' ... '95' '97' '99'
  * epi_year    (epi_year) int32 40B 2016 2017 2018 2019 ... 2022 2023 2024 2025
  * epi_week    (epi_week) int64 424B 1 2 3 4 5 6 7 8 ... 47 48 49 50 51 52 53
    number      int64 8B 0
    surface     float64 8B 0.0
Attributes: (12/31)
    GRIB_paramId:                             228
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      5313
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            accum
    ...                                       ...
    GRIB_shortName:                           tp
    GRIB_totalNumber:                         0
    GRIB_units:                               m
    long_name:                                Total precipitation
    units:                                    m
    standard_name:                            unknown

In [53]:
weekly_avg_p_dpt_df = weekly_avg_p_by_department.to_dataframe().reset_index()
weekly_avg_p_dpt_df = weekly_avg_p_dpt_df.dropna(subset=['tp']).copy()
weekly_avg_p_dpt_df

,department,epi_year,epi_week,number,surface,tp
0,05,2016,1,0,0.0,0.047247
1,05,2016,2,0,0.0,0.015412
2,05,2016,3,0,0.0,0.013858
3,05,2016,4,0,0.0,0.018778
4,05,2016,5,0,0.0,0.018379
...,...,...,...,...,...,...
16955,99,2025,49,0,0.0,0.034989
16956,99,2025,50,0,0.0,0.015804
16957,99,2025,51,0,0.0,0.002389
16958,99,2025,52,0,0.0,0.000471


In [54]:
#Original cases ds doesn't have Bogotá (Included under Cundinamarca?) 
#and CDS doesn't provide San Andrés on temp or precip cause it's too far from bounds I set (Maybe add later on?)

weekly_avg_p_dpt_df = weekly_avg_p_dpt_df[weekly_avg_p_dpt_df['department'] != '11']
weekly_avg_p_dpt_df

,department,epi_year,epi_week,number,surface,tp
0,05,2016,1,0,0.0,0.047247
1,05,2016,2,0,0.0,0.015412
2,05,2016,3,0,0.0,0.013858
3,05,2016,4,0,0.0,0.018778
4,05,2016,5,0,0.0,0.018379
...,...,...,...,...,...,...
16955,99,2025,49,0,0.0,0.034989
16956,99,2025,50,0,0.0,0.015804
16957,99,2025,51,0,0.0,0.002389
16958,99,2025,52,0,0.0,0.000471


In [55]:
weekly_temp_cases_by_department_df = weekly_temp_cases_by_department_df.drop(columns=['ANO','SEMANA','COD_DPTO_O']).copy()
weekly_temp_cases_by_department_df

,case_count,department,epi_week,epi_year,2m_temperature
0,558,76,1,2016,22.074368
1,148,68,1,2016,22.576899
2,269,73,1,2016,21.558670
3,391,05,1,2016,23.980185
4,57,50,1,2016,26.736807
...,...,...,...,...,...
16177,12,91,53,2025,24.900299
16178,4,95,53,2025,25.376633
16179,3,99,53,2025,27.129372
16180,0,94,53,2025,25.722433


In [56]:
weekly_temp_precip_cases_by_department_df = weekly_temp_cases_by_department_df.merge(weekly_avg_p_dpt_df, on=['epi_year', 'epi_week', 'department'])
weekly_temp_precip_cases_by_department_df

,case_count,department,epi_week,epi_year,2m_temperature,number,surface,tp
0,558,76,1,2016,22.074368,0,0.0,0.069697
1,148,68,1,2016,22.576899,0,0.0,0.030984
2,269,73,1,2016,21.558670,0,0.0,0.049868
3,391,05,1,2016,23.980185,0,0.0,0.047247
4,57,50,1,2016,26.736807,0,0.0,0.008278
...,...,...,...,...,...,...,...,...
16177,12,91,53,2025,24.900299,0,0.0,0.057833
16178,4,95,53,2025,25.376633,0,0.0,0.025573
16179,3,99,53,2025,27.129372,0,0.0,0.021114
16180,0,94,53,2025,25.722433,0,0.0,0.039668


In [57]:
weekly_temp_precip_cases_by_department_df = weekly_temp_precip_cases_by_department_df.drop(columns=['number','surface']).copy()
weekly_temp_precip_cases_by_department_df

,case_count,department,epi_week,epi_year,2m_temperature,tp
0,558,76,1,2016,22.074368,0.069697
1,148,68,1,2016,22.576899,0.030984
2,269,73,1,2016,21.558670,0.049868
3,391,05,1,2016,23.980185,0.047247
4,57,50,1,2016,26.736807,0.008278
...,...,...,...,...,...,...
16177,12,91,53,2025,24.900299,0.057833
16178,4,95,53,2025,25.376633,0.025573
16179,3,99,53,2025,27.129372,0.021114
16180,0,94,53,2025,25.722433,0.039668


In [58]:
len(weekly_temp_precip_cases_by_department_df[['department','epi_week','epi_year']].value_counts())

16182

In [59]:
weekly_temp_precip_cases_by_department_df.isna().value_counts()

case_count  department  epi_week  epi_year  2m_temperature  tp   
False       False       False     False     False           False    16182
Name: count, dtype: int64

In [60]:
weekly_temp_precip_cases_by_department_df['case_count'].describe()

count    16182.00000
mean        64.04919
std        154.43692
min          0.00000
25%          6.00000
50%         20.00000
75%         65.00000
max       3091.00000
Name: case_count, dtype: float64

In [61]:
weekly_temp_precip_cases_by_department_df = weekly_temp_precip_cases_by_department_df.sort_values(['department','epi_year','epi_week']).copy()
weekly_temp_precip_cases_by_department_df

,case_count,department,epi_week,epi_year,2m_temperature,tp
3,391,05,1,2016,23.980185,0.047247
34,453,05,2,2016,23.847317,0.015412
65,426,05,3,2016,24.532244,0.013858
96,420,05,4,2016,24.548456,0.018778
127,440,05,5,2016,24.743410,0.018379
...,...,...,...,...,...,...
16055,1,99,49,2025,27.481415,0.034989
16086,3,99,50,2025,27.449406,0.015804
16117,6,99,51,2025,27.745680,0.002389
16148,1,99,52,2025,27.782576,0.000471


In [62]:
def compute_x_week_rolling_avg_for_feature_in_df(window_size, rolling_feature, df):
    df[f"{rolling_feature}_{window_size}_week_avg"] = df.groupby(
        "department"
    )[rolling_feature].transform(lambda x: x.rolling(window_size).mean())

In [63]:
feature_and_window_size = [
    ("2m_temperature", 2),
    ("2m_temperature", 4),
    ("2m_temperature", 8),
    ("tp", 2),
    ("tp", 4),
    ("tp", 8),
    ("case_count", 2),
    ("case_count", 4),
    ("case_count", 8),
]

for f, s in feature_and_window_size:
    compute_x_week_rolling_avg_for_feature_in_df(s, f, weekly_temp_precip_cases_by_department_df)

weekly_temp_precip_cases_by_department_df

,case_count,department,epi_week,epi_year,2m_temperature,tp,2m_temperature_2_week_avg,2m_temperature_4_week_avg,2m_temperature_8_week_avg,tp_2_week_avg,tp_4_week_avg,tp_8_week_avg,case_count_2_week_avg,case_count_4_week_avg,case_count_8_week_avg
3,391,05,1,2016,23.980185,0.047247,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,453,05,2,2016,23.847317,0.015412,23.913751,NaN,NaN,0.031330,NaN,NaN,422.0,NaN,NaN
65,426,05,3,2016,24.532244,0.013858,24.189780,NaN,NaN,0.014635,NaN,NaN,439.5,NaN,NaN
96,420,05,4,2016,24.548456,0.018778,24.540350,24.227050,NaN,0.016318,0.023824,NaN,423.0,422.50,NaN
127,440,05,5,2016,24.743410,0.018379,24.645933,24.417857,NaN,0.018579,0.016607,NaN,430.0,434.75,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16055,1,99,49,2025,27.481415,0.034989,27.309916,27.051796,26.879162,0.042417,0.052474,0.046120,0.5,1.50,2.500
16086,3,99,50,2025,27.449406,0.015804,27.465410,27.251277,26.989419,0.025397,0.035801,0.043065,2.0,1.75,2.125
16117,6,99,51,2025,27.745680,0.002389,27.597543,27.453730,27.161203,0.009096,0.025757,0.037855,4.5,2.50,2.750
16148,1,99,52,2025,27.782576,0.000471,27.764128,27.614769,27.286244,0.001430,0.013413,0.032511,3.5,2.75,2.500


In [64]:
grouped_by_dpt = weekly_temp_precip_cases_by_department_df.groupby('department')

weekly_temp_precip_cases_by_department_df['target_next_week_cases'] = grouped_by_dpt['case_count'].shift(-1)
weekly_temp_precip_cases_by_department_df['target_epi_week'] = grouped_by_dpt['epi_week'].shift(-1)
weekly_temp_precip_cases_by_department_df['target_epi_year'] = grouped_by_dpt['epi_year'].shift(-1)

weekly_temp_precip_cases_by_department_df

,case_count,department,epi_week,epi_year,2m_temperature,tp,2m_temperature_2_week_avg,2m_temperature_4_week_avg,2m_temperature_8_week_avg,tp_2_week_avg,tp_4_week_avg,tp_8_week_avg,case_count_2_week_avg,case_count_4_week_avg,case_count_8_week_avg,target_next_week_cases,target_epi_week,target_epi_year
3,391,05,1,2016,23.980185,0.047247,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,453.0,2.0,2016.0
34,453,05,2,2016,23.847317,0.015412,23.913751,NaN,NaN,0.031330,NaN,NaN,422.0,NaN,NaN,426.0,3.0,2016.0
65,426,05,3,2016,24.532244,0.013858,24.189780,NaN,NaN,0.014635,NaN,NaN,439.5,NaN,NaN,420.0,4.0,2016.0
96,420,05,4,2016,24.548456,0.018778,24.540350,24.227050,NaN,0.016318,0.023824,NaN,423.0,422.50,NaN,440.0,5.0,2016.0
127,440,05,5,2016,24.743410,0.018379,24.645933,24.417857,NaN,0.018579,0.016607,NaN,430.0,434.75,NaN,452.0,6.0,2016.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16055,1,99,49,2025,27.481415,0.034989,27.309916,27.051796,26.879162,0.042417,0.052474,0.046120,0.5,1.50,2.500,3.0,50.0,2025.0
16086,3,99,50,2025,27.449406,0.015804,27.465410,27.251277,26.989419,0.025397,0.035801,0.043065,2.0,1.75,2.125,6.0,51.0,2025.0
16117,6,99,51,2025,27.745680,0.002389,27.597543,27.453730,27.161203,0.009096,0.025757,0.037855,4.5,2.50,2.750,1.0,52.0,2025.0
16148,1,99,52,2025,27.782576,0.000471,27.764128,27.614769,27.286244,0.001430,0.013413,0.032511,3.5,2.75,2.500,3.0,53.0,2025.0


In [65]:
weekly_temp_precip_cases_by_department_df.columns.to_list()

['case_count',
 'department',
 'epi_week',
 'epi_year',
 '2m_temperature',
 'tp',
 '2m_temperature_2_week_avg',
 '2m_temperature_4_week_avg',
 '2m_temperature_8_week_avg',
 'tp_2_week_avg',
 'tp_4_week_avg',
 'tp_8_week_avg',
 'case_count_2_week_avg',
 'case_count_4_week_avg',
 'case_count_8_week_avg',
 'target_next_week_cases',
 'target_epi_week',
 'target_epi_year']

In [84]:
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

feature_cols = [
    "2m_temperature",
    "2m_temperature_2_week_avg",
    "2m_temperature_4_week_avg",
    "2m_temperature_8_week_avg",
    "tp",
    "tp_2_week_avg",
    "tp_4_week_avg",
    "tp_8_week_avg",
    "case_count",
    "case_count_2_week_avg",
    "case_count_4_week_avg",
    "case_count_8_week_avg",
]

model_cols = feature_cols + ["department"]

required_cols = model_cols + [
    "target_next_week_cases",
    "target_epi_year",
    "target_epi_week",
]

model_df = (
    weekly_temp_precip_cases_by_department_df.dropna(subset=required_cols)
    .sort_values(["target_epi_year", "target_epi_week", "department"])
    .reset_index(drop=True)
)

train = model_df[model_df["target_epi_year"].between(2016, 2024)]
testing = model_df[model_df["target_epi_year"] == 2025]

period_ids = pd.factorize(
    pd.MultiIndex.from_frame(train[["target_epi_year", "target_epi_week"]])
)[0]

tscv = TimeSeriesSplit(n_splits=5)
cv = []

for training_indeces, validation_indeces in tscv.split(np.arange(period_ids.max() + 1)):
    training_indx = np.flatnonzero(np.isin(period_ids, training_indeces))
    validation_indx = np.flatnonzero(np.isin(period_ids, validation_indeces))
    cv.append((training_indx, validation_indx))

baseline_mae_score = np.mean(
    [
        (
            train.iloc[validation_ind]["case_count"]
            - train.iloc[validation_ind]["target_next_week_cases"]
        )
        .abs()
        .mean()
        for _, validation_ind in cv
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), feature_cols),
        ("departments", OneHotEncoder(), ["department"]),
    ]
)

pipe = Pipeline([("preprocessor", preprocessor), ("model", Ridge())])

search_ridge = GridSearchCV(
    pipe,
    {"model__alpha": np.logspace(-2, 3, 21)},
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
)

search_ridge.fit(train[model_cols], train["target_next_week_cases"])

print('Baseline MAE: ', baseline_mae_score)

print(search_ridge.best_params_)
print("Best CV MAE: ", -search_ridge.best_score_)

Baseline MAE:  9.696774193548388
{'model__alpha': np.float64(10.0)}
Best CV MAE:  9.37295981946817


In [85]:
from sklearn.metrics import mean_absolute_error

ridge_pred = search_ridge.best_estimator_.predict(testing[model_cols])

print(
    "Testing split Ridge MAE: ",
    mean_absolute_error(testing["target_next_week_cases"], ridge_pred)
)

print(
    "Testing split Baseline MAE: ",
    mean_absolute_error(testing["target_next_week_cases"], testing['case_count'])
)

Testing split Ridge MAE:  10.86694672994517
Testing split Baseline MAE:  11.12477175897748


In [72]:
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer

log_scale = make_pipeline(
    FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()
)

log_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num_log",
            log_scale,
            [
                "case_count",
                "case_count_2_week_avg",
                "case_count_4_week_avg",
                "case_count_8_week_avg",
            ],
        ),
        (
            "num",
            StandardScaler(),
            [
                "2m_temperature",
                "2m_temperature_2_week_avg",
                "2m_temperature_4_week_avg",
                "2m_temperature_8_week_avg",
                "tp",
                "tp_2_week_avg",
                "tp_4_week_avg",
                "tp_8_week_avg",
            ],
        ),
        ("departments", OneHotEncoder(drop='first'), ["department"]),
    ],
    sparse_threshold=0
)

pipe_poisson = Pipeline(
    [("preprocessor", log_preprocessor), ("poisson_model", PoissonRegressor(max_iter=2000))]
)

poisson_search = GridSearchCV(
    pipe_poisson,
    {"poisson_model__alpha": np.logspace(-5, -1, 25)},
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1
)

poisson_search.fit(train[model_cols], train['target_next_week_cases'])

print("Best Alpha value: ", poisson_search.best_params_)

print("Poisson regressor MAE: ", -poisson_search.best_score_)

Best Alpha value:  {'poisson_model__alpha': np.float64(0.021544346900318822)}
Poisson regressor MAE:  9.569355967353285


In [69]:
p_results = pd.DataFrame({
    "alpha": poisson_search.cv_results_['param_poisson_model__alpha'],
    "cv_mae" : -poisson_search.cv_results_['mean_test_score'],
    "fold_std" : poisson_search.cv_results_['std_test_score']
})

p_results = p_results.sort_values('cv_mae')
p_results

,alpha,cv_mae,fold_std
20,0.021544,9.569356,4.307130
19,0.014678,9.571774,4.302190
21,0.031623,9.577582,4.316729
18,0.010000,9.579271,4.296726
17,0.006813,9.590436,4.290233
22,0.046416,9.598954,4.331881
16,0.004642,9.603442,4.282908
15,0.003162,9.615550,4.275663
14,0.002154,9.627248,4.268434
23,0.068129,9.634151,4.358533


In [73]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import cross_val_score
import statsmodels.api as sm


class NegativeBinomialRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, fit_intercept=True, max_iters=200):
        self.fit_intercept = fit_intercept
        self.max_iters = max_iters

    def fit(self, X, y):
        X = np.asarray(X)

        if self.fit_intercept:
            X = sm.add_constant(X, has_constant="add")

        pois = sm.Poisson(y, X).fit(maxiter=self.max_iters, disp=0)
        start = np.append(pois.params, 0.5)

        self.model_ = sm.NegativeBinomial(y, X)
        self.results_ = self.model_.fit(
            start_params=start, maxiter=self.max_iters, disp=0
        )
        return self

    def predict(self, X):
        X = np.asarray(X)

        if self.fit_intercept:
            X = sm.add_constant(X, has_constant="add")

        return self.results_.predict(X)

    @property
    def alpha_(self):
        return self.results_.params[-1]


pipe_nbr = Pipeline(
    [
        ("preprocessing", log_preprocessor),
        ("nbr_model", NegativeBinomialRegressor(max_iters=2000)),
    ]
)

scores = cross_val_score(
    pipe_nbr,
    train[model_cols],
    train["target_next_week_cases"],
    cv=cv,
    scoring="neg_mean_absolute_error",
)

print("CV MAE:", -scores.mean(), "+/-", scores.std())

CV MAE: 10.220567052325771 +/- 4.839127724831565


In [77]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor

pipe_gb = Pipeline([
    ("preprocessing", log_preprocessor),
    ("model_gb", HistGradientBoostingRegressor(
        early_stopping=False,
        random_state=42,
    )),
])

param_distributions = {
    "model_gb__loss": ["absolute_error", "poisson"],
    "model_gb__learning_rate": [0.03, 0.05, 0.1],
    "model_gb__max_iter": [100, 300, 600],
    "model_gb__max_leaf_nodes": [7, 15, 31],
    "model_gb__min_samples_leaf": [10, 20, 50, 100],
    "model_gb__l2_regularization": [0.0, 1.0, 10.0],
}

search_gb = RandomizedSearchCV(
    pipe_gb,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=42,
    n_jobs=1,
)

search_gb.fit(train[model_cols], train["target_next_week_cases"])

print("Best settings:", search_gb.best_params_)
print("Best CV MAE:", -search_gb.best_score_)

Best settings: {'model_gb__min_samples_leaf': 20, 'model_gb__max_leaf_nodes': 7, 'model_gb__max_iter': 600, 'model_gb__loss': 'poisson', 'model_gb__learning_rate': 0.03, 'model_gb__l2_regularization': 1.0}
Best CV MAE: 14.528465209279812


In [88]:
import joblib
from sklearn.base import clone

final_model = clone(search_ridge.best_estimator_)

final_model.fit(model_df[model_cols], model_df["target_next_week_cases"])

joblib.dump(final_model, "artifacts/ridge_pipeline.joblib")

['artifacts/ridge_pipeline.joblib']